In [11]:
import warnings
from rdkit import RDLogger

# 屏蔽 RDKit 警告
RDLogger.DisableLog('rdApp.*')

# 或屏蔽所有 Python 警告
warnings.filterwarnings("ignore")
# 屏蔽 LightGBM 警告
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")

In [12]:
import torch
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import precision_recall_curve, auc
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
import optuna
from rdkit.Chem import Descriptors, AllChem
from tqdm import tqdm  # 导入tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold





In [13]:
# 函数：将SMILES转换为分子描述符和指纹
def smiles_to_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    # 提取描述符
    descriptors = [
        Descriptors.MolWt(mol),  # 分子量
        Descriptors.MolLogP(mol),  # LogP
        Descriptors.NumHDonors(mol),  # 氢键供体数量
        Descriptors.NumHAcceptors(mol)  # 氢键受体数量
    ]
    # 生成Morgan指纹
    fingerprint = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    fingerprint_array = np.zeros((2048,))
    Chem.DataStructs.ConvertToNumpyArray(fingerprint, fingerprint_array)
    # 合并描述符和指纹
    features = np.concatenate([descriptors, fingerprint_array])
    return features


In [14]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from tqdm import tqdm
import optuna
import numpy as np

def train_evaluate_regression_model_with_optuna(model_name, model_class, param_func, X, y, groups):
    def objective(trial):
        params = param_func(trial)
        model = model_class(**params)

        gkf = GroupKFold(n_splits=10)
        maes = []

        for train_idx, val_idx in tqdm(gkf.split(X, y, groups=groups), total=10, desc=f"Training {model_name}"):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            # ✅ 计算 MAE
            mae = mean_absolute_error(y_val, y_pred)
            maes.append(mae)

        return np.mean(maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)

    print(f'Best parameters for {model_name}: {study.best_params}')
    print(f'Best mean MAE: {study.best_value:.4f}')

In [15]:
# 数据预处理
df = pd.read_excel('../fish_unique.xlsx')
labels = df['mgperL'].values
smiles_list = df['SMILES_Canonical_RDKit'].tolist()
endpoints_a = df['endpoint']
Duration_Values_a = df['Duration_Value'].values
effects_a = df['effect']


In [16]:

features = []
new_labels = []
new_smiles_list = []
endpoints = []
Duration_Values = []
effects =[]


for smiles, label,a,b,c in zip(smiles_list, labels,Duration_Values_a,effects_a,endpoints_a):
    feature = smiles_to_features(smiles)
    if feature is not None:
        features.append(feature)
        new_labels.append(label)
        new_smiles_list.append(smiles)
        Duration_Values.append(a)
        effects.append(b)
        endpoints.append(c)

X = np.array(features)
y = np.array(new_labels)



smiles_endpoint_combined = [f"{sm}_{ep}" for sm, ep in zip(new_smiles_list, endpoints)]


groups = smiles_endpoint_combined  # 可直接用于 GroupKFold




In [17]:
def encode_column(zz):
    zz_series = pd.Series(zz)  # 转换为 Series
    unique_values = zz_series.unique()
    if len(unique_values) > 1:
        encoder = OneHotEncoder(sparse_output=False)
        return encoder.fit_transform(zz_series.values.reshape(-1, 1))
    else:
        return None  # 只有一种类别时忽略

Duration_Values =pd.Series(Duration_Values)


# 编码 effect、endpoint 和 species_group 列
effect_encoded = encode_column(effects)
endpoint_encoded = encode_column(endpoints)
#species_encoded = encode_column(df, 'species_group')

# # 将需要的列拼接成输入 X
X = np.hstack((X, Duration_Values.values.reshape(-1, 1)))

# # 拼接编码后的列（如果存在）
for encoded_feature in [effect_encoded, endpoint_encoded]:
     if encoded_feature is not None:
         X = np.hstack((X, encoded_feature))



y=np.log1p(y)

In [18]:
def xgb_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),   # L1 正则
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0)  # L2 正则
    }
from xgboost import XGBRegressor

train_evaluate_regression_model_with_optuna(
    "XGBoost",
    XGBRegressor,
    xgb_param_func,
    X, y, groups
)

[I 2025-05-16 11:46:06,207] A new study created in memory with name: no-name-49b3cc3c-6ec4-4b4a-8ad5-5446989b10a4
Training XGBoost: 100%|██████████| 10/10 [01:28<00:00,  8.88s/it]
[I 2025-05-16 11:47:35,004] Trial 0 finished with value: 0.7001216135984094 and parameters: {'n_estimators': 250, 'max_depth': 11, 'learning_rate': 0.2398688237240487, 'subsample': 0.977617504548855, 'colsample_bytree': 0.7424662764385574, 'reg_alpha': 0.4478276478242037, 'reg_lambda': 0.4384479500378047}. Best is trial 0 with value: 0.7001216135984094.
Training XGBoost: 100%|██████████| 10/10 [00:49<00:00,  4.97s/it]
[I 2025-05-16 11:48:24,759] Trial 1 finished with value: 0.7729450341958694 and parameters: {'n_estimators': 136, 'max_depth': 10, 'learning_rate': 0.2220784685319749, 'subsample': 0.687384320303811, 'colsample_bytree': 0.9818320883011842, 'reg_alpha': 0.3791395003506274, 'reg_lambda': 0.9397444660993073}. Best is trial 0 with value: 0.7001216135984094.
Training XGBoost: 100%|██████████| 10/10 [

Best parameters for XGBoost: {'n_estimators': 488, 'max_depth': 16, 'learning_rate': 0.12725570100049732, 'subsample': 0.8459667731271889, 'colsample_bytree': 0.6893360553392127, 'reg_alpha': 0.6089961074477633, 'reg_lambda': 0.3143403814059288}
Best mean MAE: 0.6483


In [19]:
from lightgbm import LGBMRegressor

def lgbm_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'verbose': -1
    }

print("Training LightGBM (Poisson)...")
train_evaluate_regression_model_with_optuna(
    "LightGBM",
    lambda **params: LGBMRegressor(objective="poisson", **params),  # ✅ 加入 Poisson 目标
    lgbm_param_func,
    X, y, groups
)

[I 2025-05-16 13:42:57,545] A new study created in memory with name: no-name-a1c2f8ef-6ca7-4243-8aaf-7fbd9117c478


Training LightGBM (Poisson)...


Training LightGBM: 100%|██████████| 10/10 [01:10<00:00,  7.04s/it]
[I 2025-05-16 13:44:07,966] Trial 0 finished with value: 0.9620203189550102 and parameters: {'n_estimators': 388, 'max_depth': 17, 'num_leaves': 209, 'learning_rate': 0.012028007597672653, 'feature_fraction': 0.8890988495550993, 'bagging_fraction': 0.6411221685554135, 'bagging_freq': 3, 'reg_alpha': 0.40303011453132687, 'reg_lambda': 0.882296102744865}. Best is trial 0 with value: 0.9620203189550102.
Training LightGBM: 100%|██████████| 10/10 [00:22<00:00,  2.24s/it]
[I 2025-05-16 13:44:30,415] Trial 1 finished with value: 0.8913417515470462 and parameters: {'n_estimators': 323, 'max_depth': 9, 'num_leaves': 215, 'learning_rate': 0.08380509879864162, 'feature_fraction': 0.6171894413710542, 'bagging_fraction': 0.9783037822955275, 'bagging_freq': 7, 'reg_alpha': 0.8845665259309815, 'reg_lambda': 0.48714924866039677}. Best is trial 1 with value: 0.8913417515470462.
Training LightGBM: 100%|██████████| 10/10 [00:13<00:00,  1.

Best parameters for LightGBM: {'n_estimators': 449, 'max_depth': 18, 'num_leaves': 236, 'learning_rate': 0.22954286574545849, 'feature_fraction': 0.8213759580992267, 'bagging_fraction': 0.8288001211611277, 'bagging_freq': 3, 'reg_alpha': 0.31677217500679644, 'reg_lambda': 0.29227251080709377}
Best mean MAE: 0.6975


In [20]:
import random

# 固定随机种子
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # for multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)  # 设置固定种子


In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import optuna
import numpy as np


class DNNWithSoftplus(nn.Module):
    def __init__(self, input_dim, hidden_sizes, activation):
        super().__init__()
        act_fn = {
            'relu': nn.ReLU(),
            'logistic': nn.Sigmoid(),
            'tanh': nn.Tanh()
        }[activation]
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev_dim, h), act_fn]
            prev_dim = h
        layers += [nn.Linear(prev_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return F.softplus(self.net(x)).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def train_dnn_with_optuna_pytorch(X, y, groups, device=device):
    def dnn_param_func(trial):
        return {
            'hidden_layer_sizes': trial.suggest_categorical(
                'hidden_layer_sizes', [(50,), (100,), (150,), (100, 50), (150, 100, 50)]
            ),
            'activation': trial.suggest_categorical('activation', ['relu', 'logistic', 'tanh']),
            'alpha': trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
            'learning_rate': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
            'optimizer': trial.suggest_categorical('solver', ['adam', 'sgd'])
        }

    def objective(trial):
        params = dnn_param_func(trial)
        model = DNNWithSoftplus(
            input_dim=X.shape[1],
            hidden_sizes=params['hidden_layer_sizes'],
            activation=params['activation']
        ).to(device)

        optimizer = {
            'adam': torch.optim.Adam,
            'sgd': torch.optim.SGD
        }[params['optimizer']](model.parameters(), lr=params['learning_rate'], weight_decay=params['alpha'])

        loss_fn = nn.MSELoss()
        gkf = GroupKFold(n_splits=10)
        fold_maes = []

        for train_idx, val_idx in gkf.split(X, y, groups=groups):
            X_train, y_train = X[train_idx], y[train_idx]
            X_val, y_val = X[val_idx], y[val_idx]

            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

            train_ds = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).float())
            train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

            model.train()
            for epoch in range(100):
                for xb, yb in train_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    pred = model(xb)
                    loss = loss_fn(pred, yb)
                    loss.backward()
                    optimizer.step()

            model.eval()
            with torch.no_grad():
                val_preds = model(torch.tensor(X_val).float().to(device)).cpu().numpy()
                mae = mean_absolute_error(y_val, val_preds)
                fold_maes.append(mae)

        return np.mean(fold_maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)
    print("\n✅ Best Parameters Found:")
    print(study.best_params)
    print(f"Mean MAE = {study.best_value:.4f}")
    return study.best_params


best_dnn_params = train_dnn_with_optuna_pytorch(X, y, groups)

[I 2025-05-16 14:23:00,583] A new study created in memory with name: no-name-a3727403-fd05-4e93-8c89-e8fc763c008c
[I 2025-05-16 14:28:05,938] Trial 0 finished with value: 0.6421984376841564 and parameters: {'hidden_layer_sizes': (100,), 'activation': 'logistic', 'alpha': 0.0012581757676676146, 'learning_rate_init': 0.00038464652600553307, 'solver': 'adam'}. Best is trial 0 with value: 0.6421984376841564.
[I 2025-05-16 14:33:42,564] Trial 1 finished with value: 0.6638280978644042 and parameters: {'hidden_layer_sizes': (150, 100, 50), 'activation': 'logistic', 'alpha': 0.0005941907862200495, 'learning_rate_init': 0.0013329134866022672, 'solver': 'adam'}. Best is trial 0 with value: 0.6421984376841564.
[I 2025-05-16 14:39:18,373] Trial 2 finished with value: 0.7896871575988215 and parameters: {'hidden_layer_sizes': (150, 100, 50), 'activation': 'tanh', 'alpha': 0.0020148035453059305, 'learning_rate_init': 0.00301341918008564, 'solver': 'adam'}. Best is trial 0 with value: 0.64219843768415


✅ Best Parameters Found:
{'hidden_layer_sizes': (100, 50), 'activation': 'relu', 'alpha': 1.0701624985063227e-05, 'learning_rate_init': 0.00011459034272400404, 'solver': 'adam'}
Mean MAE = 0.4136
